# Loading Raw atasets

In [ ]:
import pandas as pd 
import json
import os

data_parent_path= '/l/users/abdelrahman.sadallah/poetry'
datasets = {}
for folder_name in os.listdir(data_parent_path):
    folder_path = os.path.join(data_parent_path, folder_name)
    if os.path.isdir(folder_path) and folder_name != "poems_hakim":
        for file_name in os.listdir(folder_path):
            file_path = os.path.join(folder_path, file_name)
            if file_name.endswith('.csv') or file_name.endswith('.tsv'):
                print(file_path)
                datasets[folder_name] = pd.read_csv(file_path, sep='\t' if file_name.endswith('.tsv') else ',')
            elif file_name.endswith('.json'):
                with open(file_path, 'r') as f:
                    datasets[folder_name] = json.load(f)

## Loading and processing Hakim Poems

In [ ]:
## Process Hakim Poems

import os
import pandas as pd

def extract_poem_data(root_folder):
    data = []

    for genre in os.listdir(root_folder):
        genre_path = os.path.join(root_folder, genre)
        if not os.path.isdir(genre_path):
            continue

        for era in os.listdir(genre_path):
            era_path = os.path.join(genre_path, era)
            if not os.path.isdir(era_path):
                continue

            for file_name in os.listdir(era_path):
                if not (file_name.endswith(".csv") or file_name.endswith(".tsv")):
                    continue

                file_path = os.path.join(era_path, file_name)

                try:
                    with open(file_path, "r", encoding="utf-8") as f:
                        poem_text = f.read()
                except Exception as e:
                    print(f"Could not read {file_path}: {e}")
                    continue

                # Remove extension and split into poem name and poet name
                base_name = os.path.splitext(file_name)[0]
                if "_" not in base_name:
                    print(f"Skipping file with unexpected format: {file_name}")
                    continue

                # Split from the right in case poem name contains underscores
                poet_name, poem_name = base_name.rsplit("_", 1)

                data.append({
                    "poem_name": poem_name.strip(),
                    "poet_name": poet_name.strip(),
                    "era": era.strip(),
                    "genre": genre.strip(),
                    "poem": poem_text.strip()
                })

    df = pd.DataFrame(data)
    return df
root_folder = "/l/users/abdelrahman.sadallah/poetry/poems_hakim"
poems_df = extract_poem_data(root_folder)

datasets['poems_hakim'] = poems_df







## Loading Boda Scrapped Data



In [ ]:
boda_df = pd.read_csv("/path/to/data/scrapped_data/diwan_adab_poets_gate_data.csv")
datasets['boda_scrapped'] = boda_df

## Loading Rania Data

In [ ]:
adab_world = pd.read_csv('../data/scrapped_data/adab_world_final_scrapped.csv')
arapoet = pd.read_csv('../data/scrapped_data/arabic_poetry_net_scrapped.csv')
mawsooaa = pd.read_csv('../data/scrapped_data/mawsouaa_sheria.csv')
datasets['adab_world'] = adab_world
datasets['arapoet'] = arapoet
datasets['mawsooaa'] = mawsooaa


## Raw datasets statistics

In [8]:

total_count = 0
for key, dataset in datasets.items():
    if isinstance(dataset, pd.DataFrame):
        count = len(dataset)
        print(f"{key}: {count} rows")
        total_count += count
    elif isinstance(dataset, dict):
        count = len(dataset)
        print(f"{key}: {count} entries")
        total_count += count

print(f"Total: {total_count}")

FannOrFlop: 1 entries
Total: 1


# Processing Data

## Processing AraPoems

In [ ]:
datasets['AraPoems']

In [ ]:
#################### # Process AraPoems dataset
# Map English era names to Arabic equivalents
era_mapping = {
    'Pre_Islam': 'قبل الإسلام',
    'Seasoned': 'المخضرمون',
    'Abbasid': 'العصر العباسي',
    'Umayyad': 'العصر الأموي',
    'Andalusian': 'العصر الأندلسي',
    'Mamluk': 'العصر المملوكي',
    'Islamic': 'العصر الإسلامي',
    'Ayyubid': 'العصر الأيوبي',
    'Fatimid': 'العصر الفاطمي',
    'Ottoman': 'العصر العثماني',
    'Modern': 'العصر الحديث',
    'Dual_eras': 'عصرين',
    'Unspecified': 'غير محدد',
}

# Apply the mapping to the 'era' column
datasets['AraPoems']['era'] = datasets['AraPoems']['era'].map(era_mapping)

In [ ]:
############ Number of unique poem_names
unique_poem_names = datasets['AraPoems']['poem_title'].nunique()

print(f"Number of unique poem names: {unique_poem_names}")

In [15]:
# Copy the original dataframe
df = datasets['AraPoems'].copy()

# Ensure hemistichs are strings and fill NaNs
df['first_hemistich'] = df['first_hemistich'].fillna('').astype(str)
df['second_hemistich'] = df['second_hemistich'].fillna('').astype(str)

# Create the poem line format: first\tsecond
df['formatted_line'] = df['first_hemistich'] + '\t' + df['second_hemistich']

# Group by poet and poem title, and join lines
poem_texts = df.groupby(['poem_title', 'poet'])['formatted_line'].apply('\n'.join).reset_index(name='poem_text')

# Keep only one row per poem (first one), then merge back the poem_text
first_rows = df.drop_duplicates(subset=['poem_title', 'poet'])

# Merge back the full poem text
final_df = pd.merge(first_rows, poem_texts, on=['poem_title', 'poet'], how='inner')

# Drop the temporary 'formatted_line' column
final_df = final_df.drop(columns=['formatted_line'])

# Store result back in datasets['AraPoems']
datasets['AraPoems'] = final_df

KeyError: 'AraPoems'

In [ ]:
datasets['AraPoems']['poem_text']

0         إيها جداب سيد الأعراب\tيا معدن الطعان والضراب\...
1         إن الجنود حثها طلابها\tوالأرقميون فذا شهابها\n...
2         ليس للعجم نصرة في عشيري\tإن أراد الطميح نجل ال...
3              إن نصر الطميح أكرم نصر\tوحنو على بني الأعمام
4         احمل ظليم في العجاج الأسود\tففيه عرو كالهزبر ا...
                                ...                        
165504            تروح كأنها مما أصابت\tمعلقة بأحقيها الدلي
165505        وكل مكارم الأخلاق صارت\tإليه همتي وبه اكتسابي
165506     بأنا قد قتلنا الخير قرطا\tوجلنا في سراة بني نمير
165507     إذا اكتنفا بضرهما سقيما\tيعادى الداء ليس له مقيت
165508    ابت لي عفتي وأبى بلائي\tوأخذي الحمد بالثمن الربيح
Name: poem_text, Length: 165509, dtype: object

In [ ]:
def unify_arapoems(df: pd.DataFrame) -> pd.DataFrame:

    """Transform AraPoems dataset to unified schema."""
    df = df.copy()

    # Combine hemistiches into 'poem_text'
    # df['poem_text'] = df['first_hemistich'].astype(str) + '\n' + df['second_hemistich'].astype(str)

    # Merge 'البحر' and 'جزء البحر' into 'meter'
    df['meter'] = df['البحر'].astype(str) + '-' + df['جزء البحر'].astype(str)

    # Rename and drop as requested
    df.rename(columns={
        'poem_title': 'poem_title',
        'poet': 'poet_name',
        'era': 'poet_era',
        'قافية': 'rhyme',
        'type_ar': 'genre',
        'link': 'source',
        
    }, inplace=True)

    # Drop unwanted columns
    df.drop(columns=[
        'first_hemistich', 'second_hemistich',
        'البحر', 'جزء البحر',
        'meter', 'sub_meter',
        'rhyme', 'type_en','gender'

    ], errors='ignore', inplace=True)

    return df

## Processing Arabic PCD

In [ ]:
datasets['Arabic PCD']

,العصر,الشاعر,الديوان,القافية,البحر,الشطر الايسر,الشطر الايمن,البيت
0,قبل الإسلام,عمرو بنِ قُمَيئَة,الديوان الرئيسي,د,الطويل,وَأَن تَجمَعا شَملي وَتَنتَظِرا غَدا,خَليلَيَّ لا تَستَعجِلا أَن تَزَوَّدا,خَليلَيَّ لا تَستَعجِلا أَن تَزَوَّدا وَأَن...
1,قبل الإسلام,عمرو بنِ قُمَيئَة,الديوان الرئيسي,د,الطويل,وَلا سُرعَتي يَوماً بِسابِقَةِ الرَدى,فَما لَبَثٌ يَوماً بِسابِقٍ مَغنَمٍ,فَما لَبَثٌ يَوماً بِسابِقٍ مَغنَمٍ وَلا سُ...
2,قبل الإسلام,عمرو بنِ قُمَيئَة,الديوان الرئيسي,د,الطويل,وَتَستَوجِبا مَنّاً عَلَيَّ وَتُحمَدا,وَإِن تُنظِراني اليَومَ أَقضِ لُبانَةً,وَإِن تُنظِراني اليَومَ أَقضِ لُبانَةً وَتَ...
3,قبل الإسلام,عمرو بنِ قُمَيئَة,الديوان الرئيسي,د,الطويل,تُؤامِرُني سِرّاً لِأَصرِمَ مَرثَدا,لَعَمرُكَ ما نَفسٌ بِجِدٍ رَشيدَةٍ,لَعَمرُكَ ما نَفسٌ بِجِدٍ رَشيدَةٍ تُؤامِرُ...
4,قبل الإسلام,عمرو بنِ قُمَيئَة,الديوان الرئيسي,د,الطويل,وَأَفرَعَ في لَومي مِراراً وَأَصعَدا,وَإِن ظَهَرَت مِنهُ قَوارِصُ جَمَّةٌ,وَإِن ظَهَرَت مِنهُ قَوارِصُ جَمَّةٌ وَأَفر...
...,...,...,...,...,...,...,...,...
1831765,الحديث,شهاب غانم,شهاب غانم,ن,الخفيف,وأحلى قصيدة تَتَغنى,هي أغلى ما أنشأ اللَّه في الدنيا,هي أغلى ما أنشأ اللَّه في الدنيا وأحلى قصيد...
1831766,الحديث,شهاب غانم,شهاب غانم,ن,الخفيف,كحلم يغشى الجفون الوسنى,هي أغرودة الأغاريد تنساب,هي أغرودة الأغاريد تنساب كحلم يغشى الجفون ا...
1831767,الحديث,شهاب غانم,شهاب غانم,ن,الخفيف,يتداعى وجداً ويخفق حسنا,هي شلال بهجة وبهاء,هي شلال بهجة وبهاء يتداعى وجداً ويخفق حسنا
1831768,الحديث,شهاب غانم,شهاب غانم,ن,الخفيف,يدك الحدود سجناً فسجنا,هي حلم الهوى ومنطلقي الباقي,هي حلم الهوى ومنطلقي الباقي يدك الحدود سجنا...


In [ ]:
def unify_arabic_pcd(df: pd.DataFrame) -> pd.DataFrame:
    """Transform Arabic PCD dataset to unified schema."""
    df = df.copy()
    
    # Create 'poem_text' as right hemistich + newline + left hemistich
    df['poem_text'] = df['الشطر الايمن'].astype(str) + '\t' + df['الشطر الايسر'].astype(str)
    
    # Rename columns
    df.rename(columns={
        'الشاعر': 'poet_name',
        'العصر': 'poet_era',
        'القافية': 'rhyme',
        'البحر': 'meter',
    }, inplace=True)

    # Drop unused columns
    df.drop(columns=['الديوان', 'الشطر الايمن', 'الشطر الايسر', 'البيت'], errors='ignore', inplace=True)

    return df


## Processing Arabic Poetry Dataset (6th - 21st century)

In [ ]:
import random

datasets['Arabic Poetry Dataset (6th - 21st century)']
# Sample a random row from the dataset
random_row = datasets['Arabic Poetry Dataset (6th - 21st century)'].sample(n=1)

# Display the poem_text of the sampled row
print(random_row['poem_text'].iloc[0])

اجلت احلامي سنين اجلت احلامي التي يوما عرفناها معا اجلت ما لا تعرفين اجلت احلام السنين جميعها وجلست وحدي انتظر فلربما تاتين قد كان عهد بينا خنت العهود وقد حنثت الان عمدا باليمين يا ايها الحلم المءجل في دمي حتي متي ابقي علي حالي وانت تءجلين قلت انتظرني وانتظرت بلا امل لم داءما عكس اتجاهي في الفضاء تسافرين زينت قلبي بالمني وفرشت احلامي بزهر الياسمين اجلت احلامي سنين وتكسر الاحساس مات علي مشارف ما فعلت وتفعلين *** عمري مضي احلي سنين العمر تمرق من امامي ذاهبة ليس الهوي منا من الاحباب او حتي هبة عمري مضي ومتي سالتك قلت لي متاهبة وتركت لي جرحا تلوث في فءادي من حصار الاتربة من قال ان الحب يشفي بالرقي او من كلام الاحجبة ماذا ستنتظرين قولي ربما الان اقتنع بصمت الاجوبة لا تنظري لافق اني لم اعد ابدا اصدق ان تكوني راهبة كذبت نفسي كي لا اصدق ان تكوني كاذبة كم كنت في صمتي اذوب اذا نظرت معاتبة قد كانت الدنيا بعيني لا تساوي اي شيء لو رايتك غاضبة كنت امد بداخلي دوما يدي اتحسك فاراك دوما داخلي متشعبة في القلب انت وفي العيون وفي دماءي ذاءبة ارجوك عودي بالحياة الصاخبة انا ذقت احلي ما عرفت علي يديك من الهو

In [ ]:
datasets['Arabic Poetry Dataset (6th - 21st century)']['poem_text']

0        عيناك غابتا نخيل ساعة السحر او شرفتان راح يناي...
1         انا لا ازال و في يدي قدحي ياليل اين تفرق الشر...
2         علي مقلتيك ارتشفت النجوم وعانقت امالي الايبة ...
3        اساطير من حشرجات الزمان نسيج اليد البالية رواه...
4        والتف حولك ساعداي ومال جيدك في اشتهاء كالزهرة ...
                               ...                        
58016    لروح صهيل لا تحويه الاوقات ذنبك انك تمتد علي ا...
58017    اه لو اني ابني الشمس بعيني من طين هواء وسراب م...
58018    في عينيك يا امي لماذا الدمع منتظم كعقد الءلء ا...
58019    النوم يوقظ طرفي الظامي علي لحن تموج من بعيد ار...
58020    السلم الذي نزلت فيه لسماء سلم من الجنان جاء او...
Name: poem_text, Length: 58021, dtype: object

In [ ]:

# ---------- Dataset 6: Arabic Poetry Dataset (6th - 21st century) ----------
def unify_century_dataset(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Combine 'poet location' and 'poem language type' into 'tags'
    # df['tags'] = df['poem_style'].astype(str) + ', ' + df['poet_cat'].astype(str)
    # Update the poet_cat column values based on the condition
    df['poet_era'] = df['poet_cat'].apply(lambda x: x if isinstance(x, str) and "العصر" in x else None)
    df['location'] = df['poet_cat'].apply(lambda x: x if isinstance(x, str) and "العصر" not in x else None)
    # Drop the original columns
    df.drop(columns=['poem_style', 'poet_cat','poet_id'], errors='ignore', inplace=True)
    df.rename(columns={
        'poem_style': 'langauge',
        'poem_link': 'source',
        'poem_text': 'poem_text',
        'poem_title': 'poem_title',
        'poet_link': 'poet_url',
        'poet_name': 'poet_name'
    }, inplace=True)
    return df

## Processing Ashaar

In [ ]:
import re
import pandas as pd

# Step 1: Extract (line, single_quoted_count) for each line
lines_with_single_quotes = []

for poem in datasets['Ashaar']['poem verses']:
    if isinstance(poem, str):
        for line in poem.split('\n'):
            single_quotes = re.findall(r"'(.*?)'", line)
            count = len(single_quotes)
            if count > 0:
                lines_with_single_quotes.append((line, count, single_quotes))

# Step 2: Create a DataFrame
df_single_quotes = pd.DataFrame(lines_with_single_quotes, columns=['line', 'single_quoted_count', 'quoted_parts'])

# Step 3: Show value counts
value_counts = df_single_quotes['single_quoted_count'].value_counts().sort_index()
print("Value Counts (Number of Single-Quoted Parts per Line):\n")
print(value_counts)

# Step 4: Sample one line for each count
samples = df_single_quotes.groupby('single_quoted_count').apply(lambda g: g.sample(1, random_state=42)).reset_index(drop=True)

# Step 5: Display samples
for _, row in samples.iterrows():
    print(f"\nSingle-Quoted Parts: {row['single_quoted_count']}")
    print(f"Sample Line: {row['line']}")
    print(f"Extracted Parts: {row['quoted_parts']}")


Value Counts (Number of Single-Quoted Parts per Line):

single_quoted_count
1     2321846
2     1967891
3      301119
4       65361
5       11597
6        2550
7         467
8         140
9          41
10         21
11          2
14          1
15          1
19          1
Name: count, dtype: int64

Single-Quoted Parts: 1
Sample Line:  'مُــتَــعَــمِّمــٌ بِــالشَــرِّ مُـؤتَـزِرٌ بِهِ'
Extracted Parts: ['مُــتَــعَــمِّمــٌ بِــالشَــرِّ مُـؤتَـزِرٌ بِهِ']

Single-Quoted Parts: 2
Sample Line:  'ترى الزَّعفرانَ سقى خدَّها' 'مجاجتهُ والعبيرَ المدوفا'
Extracted Parts: ['ترى الزَّعفرانَ سقى خدَّها', 'مجاجتهُ والعبيرَ المدوفا']

Single-Quoted Parts: 3
Sample Line:  '... فلا دُمتِ يا نعمة القافيهْ !!).' '1414هـ' '1993م']
Extracted Parts: ['... فلا دُمتِ يا نعمة القافيهْ !!).', '1414هـ', '1993م']

Single-Quoted Parts: 4
Sample Line:  '***' 'هي امرأةٌ' 'مِن الخمرِ المُذابِ تُذابْ' 'بقلبٍ في هواها ذابْ'
Extracted Parts: ['***', 'هي امرأةٌ', 'مِن الخمرِ المُذابِ تُذابْ', 'بقلبٍ في هواها ذابْ']



/tmp/ipykernel_2971615/888656296.py:24: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  samples = df_single_quotes.groupby('single_quoted_count').apply(lambda g: g.sample(1, random_state=42)).reset_index(drop=True)


In [ ]:
import re
import pandas as pd

def process_poem_row(text):
    if not isinstance(text, str):
        return None
    lines = text.split('\n')
    processed_lines = []
    for line in lines:
        parts = re.findall(r"'(.*?)'", line)
        
        # If any line has more than 8 parts, drop the entire row
        if len(parts) > 8:
            return None
        # If no parts found, skip this line
        if not parts:
            continue

        # Split parts into two halves
        mid = len(parts) // 2
        first_half = ''.join(parts[:mid])
        second_half = ''.join(parts[mid:])
        processed_line = f"{first_half}\t{second_half}"
        processed_lines.append(processed_line)
    return '\n'.join(processed_lines) if processed_lines else None

# Apply the transformation
datasets['Ashaar']['poem verses'] = datasets['Ashaar']['poem verses'].apply(process_poem_row)

# Drop rows where processing failed (e.g., lines with >8 parts)
datasets['Ashaar'].dropna(subset=['poem verses'], inplace=True)


In [ ]:

###############################3 Cleaning Ashaar Explnanaion column ###################

def clean_poem_description(desc_str):
    if not isinstance(desc_str, str) or not desc_str.strip().startswith('['):
        return None

    # Try to fix common issue: missing closing bracket
    desc_str = desc_str.strip()
    if not desc_str.endswith(']'):
        desc_str += ']'

    # First attempt: use ast.literal_eval for Python-style strings
    try:
        data_list = ast.literal_eval(desc_str)
        if isinstance(data_list, list):
            text_parts = [item.get("text", "") for item in data_list if isinstance(item, dict)]
            return " ".join(text_parts).strip()
    except Exception as e1:
        pass  # fall through to json-based fallback

    # Fallback attempt: convert to JSON-style string
    def safe_to_json(s):
        s = re.sub(r"(?<![a-zA-Z])'([^']*?)':", r'"\1":', s)  # keys
        s = re.sub(r':\s*\'(.*?)\'(,?)', r': "\1"\2', s)      # string values
        return s

    try:
        json_like = safe_to_json(desc_str)
        data_list = json.loads(json_like)
        if isinstance(data_list, list):
            text_parts = [item.get("text", "") for item in data_list if isinstance(item, dict)]
            return " ".join(text_parts).strip()
    except Exception as e2:
        return None

    return None

# Apply the function to the column
datasets['Ashaar']['cleaned_poem_description'] = datasets['Ashaar']['poem description'].apply(clean_poem_description)

# Count successfully processed rows
num_processed = datasets['Ashaar']['cleaned_poem_description'].apply(lambda x: isinstance(x, str) and x.strip() != "").sum()

print(f"Number of successfully processed rows: {num_processed}")

number_of_original_rows = len(datasets['Ashaar']['poem description'].dropna())
print(f"Number of original rows: {number_of_original_rows}")

# Replace the old column 'poem description' with the new column 'cleaned_poem_description'
datasets['Ashaar']['poem description'] = datasets['Ashaar']['cleaned_poem_description']

# Drop the 'cleaned_poem_description' column
datasets['Ashaar'].drop(columns=['cleaned_poem_description'], inplace=True)
################################################################################################33

########################################### Cleaning 

Number of successfully processed rows: 0
Number of original rows: 14468


In [ ]:
datasets['Ashaar'] ##############################################

,poem title,poem meter,poem verses,poem theme,poem url,poet name,poet description,poet url,poet era,poet location,poem description,poem language type
0,أصبح الملك للذي فطر الخلق,بحر الخفيف,أَصبَحَ المُلك لِلَّذي فَطر الخَل\tقَ بِتَقدير...,قصيدة دينية,https://www.aldiwan.net/poem16182.html,الامير منجك باشا,منجك بن محمد بن منجك بن ابي بكر بن عبد القادر ...,https://www.aldiwan.net/cat-poet-alamir-mnczyk...,العصر العثماني,NaN,None,NaN
1,من أي مولى ارتجي,بحر مجزوء الرمل,مِن أَي مَولى اِرتَجي\tوَلاي باب التَجيوَاللَه...,قصيدة دينية,https://www.aldiwan.net/poem16183.html,الامير منجك باشا,منجك بن محمد بن منجك بن ابي بكر بن عبد القادر ...,https://www.aldiwan.net/cat-poet-alamir-mnczyk...,العصر العثماني,NaN,None,NaN
2,العبد عبدك يا من أنت سيده,بحر البسيط,العَبد عَبدك يا مَن أَنتَ سَيدهُ\tوَلَيسَ غَير...,قصيدة ذم,https://www.aldiwan.net/poem16184.html,الامير منجك باشا,منجك بن محمد بن منجك بن ابي بكر بن عبد القادر ...,https://www.aldiwan.net/cat-poet-alamir-mnczyk...,العصر العثماني,NaN,None,NaN
3,لو كنت أطمع بالمنام توهما,بحر الكامل,لَو كُنتَ أَطمَع بِالمَنام تَوهما\tلَسالَت طَي...,قصيدة عامه,https://www.aldiwan.net/poem16185.html,الامير منجك باشا,منجك بن محمد بن منجك بن ابي بكر بن عبد القادر ...,https://www.aldiwan.net/cat-poet-alamir-mnczyk...,العصر العثماني,NaN,None,NaN
4,يعد علي أنفاسي ذنوبا,بحر الوافر,يعد عَليَّ أَنفاسي ذُنوباً\tإِذا ما قُلت أَفدي...,قصيدة عامه,https://www.aldiwan.net/poem16186.html,الامير منجك باشا,منجك بن محمد بن منجك بن ابي بكر بن عبد القادر ...,https://www.aldiwan.net/cat-poet-alamir-mnczyk...,العصر العثماني,NaN,None,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
254625,NaN,المضارع,وعندنا ليس محض\tإلى الشرق أن تولي,NaN,NaN,dataset collectors,NaN,NaN,NaN,NaN,None,فصيح
254626,NaN,المضارع,من الليالي نهار\tومن عتمها ضياء,NaN,NaN,dataset collectors,NaN,NaN,NaN,NaN,None,فصيح
254627,NaN,المضارع,مقامه إذ تجلى\tوفي البيد إذ تحلى,NaN,NaN,dataset collectors,NaN,NaN,NaN,NaN,None,فصيح
254628,NaN,المضارع,على بابك انتظرنا\tوفي البال ألف حيلةمن الوصل ل...,NaN,NaN,dataset collectors,NaN,NaN,NaN,NaN,None,فصيح


In [ ]:
# ---------- Dataset 5: Ashaar ----------
def unify_ashaar(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Drop the original columns
    df.drop(columns=['poet location', 'poem language type'], errors='ignore', inplace=True)
    df.rename(columns={
        'poet location': 'location',
        'poem language type': 'language',
        'poem title': 'poem_title',
        'poem meter': 'meter',
        'poem verses': 'poem_text',
        'poem theme': 'genre',
        'poem url': 'poem_url',
        'poet name': 'poet_name',
        'poet description': 'poet_description',
        'poet url': 'poet_url',
        'poet era': 'poet_era',
        'poem description': 'overall_explanation',
    }, inplace=True)
    return df

## Process FannOrFlop

In [ ]:
datasets['FannOrFlop']

,source,title,tags,verse_count,author,era,meter,genre,id,explanation,poem_verses,raw_explanation
0,https://arabic-poetry.net/poem/20236-حياكم-الل...,حَيّاكُمُ اللَهُ أَحيوا العِلمَ وَالأَدَبا,['البسيط' 'سياسية' 'العصر الحديث'],40,حافظ ابراهيم,العصر الحديث,البسيط,سياسية,poem_1,[{'explanation': 'يبدأ الشاعر بتهنئة مخاطبيه، ...,1\n\nحَيّـاكُمُ اللَـهُ أَحيـوا العِلمَ وَالأَ...,Verses 1-12:\nبسم الله الرحمن الرحيم،\n\nهذه ق...
1,https://arabic-poetry.net/poem/20345-غاب-الأدي...,غابَ الأَديبُ أَديبُ مِصرٍ وَاِختَفى,['الكامل' 'رثاء' 'العصر الحديث'],3,حافظ ابراهيم,العصر الحديث,الكامل,رثاء,poem_2,[{'explanation': 'يبدأ الشاعر ببيان غياب الأدي...,1\n\nغـابَ الأَديـبُ أَديـبُ مِصرٍ وَاِختَفى\n...,Verses 1-3:\nهذه القصيدة رثاء في أديب مصري، يم...
2,https://arabic-poetry.net/poem/20152-عثمان-إنك...,عُثمانُ إِنَّكَ قَد أَتَيتَ مُوَفَّقاً,['الكامل' 'مدح' 'العصر الحديث'],3,حافظ ابراهيم,العصر الحديث,الكامل,مدح,poem_3,[{'explanation': 'يبدأ الشاعر بمدح عثمان بن عف...,1\n\nعُثمـانُ إِنَّكَ قَد أَتَيتَ مُوَفَّقاً\n...,Verses 1-3:\nهذه القصيدة مدحٌ لعُثمان بن عفان ...
3,https://arabic-poetry.net/poem/20188-إن-عضيك-ي...,إِنَّ عَضّيكَ يا أَخي بِالمَلامِ,['الخفيف' 'عتاب' 'العصر الحديث'],10,حافظ ابراهيم,العصر الحديث,الخفيف,عتاب,poem_4,[{'explanation': 'يستهل الشاعر قصيدته بتوبيخ أ...,1\n\nإِنَّ عَضــّيكَ يــا أَخــي بِــالمَلامِ\...,Verses 1-10:\nتتحدث هذه القصيدة عن لوعة الشاعر...
4,https://arabic-poetry.net/poem/20185-من-واجد-م...,مِن واجِدٍ مُنَقِّرِ المَنامِ,['الرجز' 'عتاب' 'العصر الحديث'],15,حافظ ابراهيم,العصر الحديث,الرجز,عتاب,poem_5,[{'explanation': 'يصف الشاعر نفسه بأنه شخصٌ تع...,1\n\nمِن واجِدٍ مُنَقِّرِ المَنامِ\n\nطَريدَ د...,Verses 1-12:\nتتحدث هذه القصيدة عن شاعرٍ مُتعب...
...,...,...,...,...,...,...,...,...,...,...,...,...
6979,https://arabic-poetry.net/poem/4547-إليك-ما-أن...,إِلَيكِ ما أَنا مِن لَهوٍ وَلا طَرَبِ,['البسيط' 'صبر' 'العصر العباسي'],17,البُحتُرِيّ,العصر العباسي,البسيط,صبر,poem_6882,[{'explanation': 'يُخاطِب الشاعر جهةً ما (ربما...,1\n\nإِلَيـكِ مـا أَنـا مِـن لَهـوٍ وَلا طَرَب...,Verses 1-12:\nهذه القصيدة تعبر عن شاعر يواجه ص...
6980,https://arabic-poetry.net/poem/4474-خذ-العيش-ا...,خذِ العيشَ الهنيَّ من المجوس,['الوافر' 'غزل' 'العصر العباسي'],15,أبو نُوّاس,العصر العباسي,الوافر,غزل,poem_6835,[{'explanation': 'يبدأ الشاعر بدعوة إلى التمتع...,1\n\nخـذِ العيـشَ الهنيَّ من المجوس\n\nمعــاقر...,Verses 1-12:\nتتحدث هذه القصيدة عن متعة الشراب...
6981,https://arabic-poetry.net/poem/3786-اصبر-لمر-ح...,اِصبِر لِمَرِّ حَوادِثِ الدَهرِ,['الكامل' 'صبر' 'العصر العباسي'],11,أبو نُوّاس,العصر العباسي,الكامل,صبر,poem_6891,[{'explanation': 'يبدأ الشاعر بنصيحةٍ بالصبر ع...,1\n\nاِصـبِر لِمَـرِّ حَـوادِثِ الـدَهرِ\n\nفَ...,Verses 1-11:\nتتحدث هذه القصيدة عن الموت والحس...
6982,https://arabic-poetry.net/poem/3146-لخير-إمام-...,لِخَيرِ إِمامٍ قامَ مِن خَيرِ مَعشَرٍ,['الطويل' 'سياسية' 'العصر العباسي'],10,أبو العَتاهِيَة,العصر العباسي,الطويل,سياسية,poem_6901,[{'explanation': 'يبدأ الشاعر بمدح الخليفة الم...,1\n\nلِخَيـرِ إِمـامٍ قـامَ مِن خَيرِ مَعشَرٍ\...,Verses 1-10:\nهذه قصيدة تُعبر عن شكوى وتضرع إل...


In [ ]:
import pandas as pd
import re

def unify_fann_or_flop(df: pd.DataFrame):
    df = df.copy()

    df.rename(columns={
        'source': 'source',
        'title': 'poem_title',
        'tags': 'tags',
        'verse_count': 'verse_count',
        'author': 'poet_name',
        'era': 'poet_era',
        'meter': 'meter',
        'genre': 'genre',
        'id': 'poem_id',
        'explanation': 'overall_explanation',
        'poem_verses': 'poem_text',
        'raw_explanation': 'verses_explanation'
    }, inplace=True)

    def extract_verses(text):
        if not isinstance(text, str):
            return []

        # Split into blocks using verse number markers
        blocks = re.split(r'\n*\b\d+\b\n*', text)
        blocks = [block.strip() for block in blocks if block.strip()]

        verses = []
        for block in blocks:
            lines = [line.strip() for line in block.split('\n') if line.strip()]
            if len(lines) >= 2:
                verses.append('\t'.join(lines))
            elif len(lines) == 1:
                verses.append(lines[0])
        return verses

    def process_tags(tags: str):
        """
        Process the tags string to extract meter, genre, and poet_era.

        Returns:
            tuple: (meter, genre, poet_era) if valid, else None.
        """
        if not isinstance(tags, str):
            return None

        # Remove surrounding brackets if present
        tags = tags.strip().lstrip('[').rstrip(']')

        # Extract values enclosed in single or double quotes
        parts = re.findall(r"'([^']+)'|\"([^\"]+)\"", tags)

        # Flatten the tuple pairs returned by re.findall
        parts = [p1 or p2 for p1, p2 in parts]

        if len(parts) != 3:
            return None

        return tuple(part.strip() for part in parts)

    valid_rows = []
    invalid_rows = []

    for idx, row in df.iterrows():
        verses = extract_verses(row['poem_text'])
        expected_count = pd.to_numeric(row['verse_count'], errors='coerce')

        ### this is commented because after analysis, we found that there are 67 samples with wrong reported verses.
        # # Validate verse count
        # if pd.isna(expected_count) or expected_count != len(verses):
        #     invalid_rows.append(row)
        #     continue

        # Process tags
        processed_tags = process_tags(row.get('tags', ''))
        if processed_tags is None:
            invalid_rows.append(row)
            continue

        meter, genre, poet_era = processed_tags

        # Update row with processed data
        row['poem_text'] = '\n'.join(verses)
        row['meter'] = meter
        row['genre'] = genre
        row['poet_era'] = poet_era

        valid_rows.append(row)


    valid_df = pd.DataFrame(valid_rows).reset_index(drop=True)
    invalid_df = pd.DataFrame(invalid_rows).reset_index(drop=True)

    print(len(valid_df), "valid rows")
    print(len(invalid_df), "invalid rows")

    # Remove the 'verse_count' column from the dataframe
    valid_df.drop(columns=['verse_count'], inplace=True, errors='ignore')

    return valid_df


## Process Arabic Poetry Dataset

In [ ]:
datasets['Arabic Poetry Dataset']

,poet_name,poet_era,poem_tags,poem_title,poem_text,poem_count
0,لقيط بن يعمر الإيادي,العصر الجاهلي,"قصائد هجاء, عموديه, بحر الوافر, قافية الدال (د)",سلام في الصحيفة من لقيط,سَلامٌ في الصَحيفَةِ مِن لَقيطٍ\nإِلى مَن بِال...,عدد الابيات : 4
1,لقيط بن يعمر الإيادي,العصر الجاهلي,"قصائد حزينه, عموديه, بحر البسيط, قافية الألف ...",يا دار عمرة من محتلها الجرعا,يا دارَ عَمرَةَ مِن مُحتَلِّها الجَرَعا\nهاجَت...,عدد الابيات : 60
2,لقيط بن يعمر الإيادي,العصر الجاهلي,"قصائد قصيره, عموديه, بحر الرجز, قافية الألف (ا)",وخاننا خوان في ارتباعنا,وَخانَنا خَوّانٌ في اِرتِباعِنا\nفَاِنفَدَّ لِ...,عدد الابيات : 1
3,زبان بن سيار الفزاري,العصر الجاهلي,"قصائد قصيره, عموديه, بحر الطويل, قافية الألف ...",تنح إليكم يا ابن كوز فإننا,تَنَحَّ إِلَيكُم يا اِبنَ كوزٍ فَإِنَّنا\nوَإِ...,عدد الابيات : 1
4,زبان بن سيار الفزاري,العصر الجاهلي,"قصائد قصيره, عموديه, بحر الطويل, قافية الباء ...",تطارحه الأنساب حتى رددنه,تُطارِحُهُ الأَنسابُ حَتّى رَدَدنَهُ\nإِلى نَس...,عدد الابيات : 1
...,...,...,...,...,...,...
75017,أبو شراعة,العصر العباسي,"قصائد حزينه, عموديه, بحر الخفيف, قافية اللام ...",تلوم جودي لبرمة الطفشيل,تَلومُ جودي لِبُرمَةِ الطَفشيلِ\nوَاِستَهِلّي ...,عدد الابيات : 9
75018,أبو شراعة,العصر العباسي,"قصائد رومنسيه, عموديه, بحر البسيط, قافية اللا...",وردت دار سعيد وهي خالية,وَرَدتُ دارَ سَعيدٍ وَهيَ خالِيَةٌ\nوَكانَ أَب...,عدد الابيات : 4
75019,أبو شراعة,العصر العباسي,"قصائد قصيره, عموديه, بحر الطويل, قافية الياء ...",ألا لا أبالي في العلى ما أصابني,أَلا لا أُبالي في العُلى ما أَصابَني\nوَإِن نَ...,عدد الابيات : 3
75020,أبو شراعة,العصر العباسي,"قصائد عتاب, عموديه, بحر الوافر, قافية الياء (ي)",بني سوار إن رثت ثيابي,بَني سَوّارَ إِن رَثَت ثِيابي\nوَكَلَّ عَنِ ال...,عدد الابيات : 9


In [ ]:
print(datasets['Arabic Poetry Dataset']['poem_text'].iloc[0])

سَلامٌ في الصَحيفَةِ مِن لَقيطٍ
إِلى مَن بِالجَزيرَةِ مِن إِيادِ
بِأَنَّ اللَيثَ كِسرى قَد أَتاكُمُ
فَلا يَشغَلكُمُ سَوقُ النِقادِ
أَتاكُم مِنهُمُ سِتّونَ أَلفاً
يَزُجّونَ الكَتائِبَ كَالجَرادِ
عَلى حَنَقٍ أَتَيناكُم فَهَذا
أَوانُ هَلاكِكُم كَهَلاكِ عادِ



In [ ]:
import pandas as pd
import re

def unify_arabic_poetry_dataset(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Rename columns
    df.rename(columns={
        'poet_name': 'poet_name',
        'poet_era': 'poet_era',
        'poem_tags': 'tags',
        'poem_title': 'poem_title',
        'poem_text': 'poem_text',
        'poem_count': 'poem_verses'
    }, inplace=True)

    # Extract number from poem_verses
    df['poem_verses'] = df['poem_verses'].astype(str).str.extract(r'(\d+)').astype(float).astype('Int64')

    # Extract tags into genre, poem_type, meter, rhyme
    def extract_tags(tags):
        if pd.isna(tags):
            return [None, None, None, None]
        parts = [t.strip() for t in str(tags).split(',')]
        if len(parts) == 4:
            return parts
        return [None, None, None, None]

    extracted_tags = df['tags'].apply(extract_tags).apply(pd.Series)
    df[['genre', 'poem_type', 'meter', 'rhyme']] = extracted_tags

    # Report failed tag extractions
    failed_tags_mask = extracted_tags.isnull().any(axis=1)
    failed_tags = df.loc[failed_tags_mask, 'tags']
    print(f"Number of rows with invalid or missing poem_tags: {len(failed_tags)}")
    print("Examples of failed tags:")
    print(failed_tags.dropna().drop_duplicates().head(10).to_list())

    # Format poem_text into paired half-verses
    def format_poem_text(text):
        if pd.isna(text):
            return text
        lines = [line.strip() for line in str(text).split('\n') if line.strip()]
        paired_lines = []
        for i in range(0, len(lines) - 1, 2):
            paired_lines.append(f"{lines[i]}\t{lines[i+1]}")
        return '\n'.join(paired_lines)

    df['poem_text'] = df['poem_text'].apply(format_poem_text)

    # Calculate actual verse count
    def count_verses(text):
        if pd.isna(text):
            return 0
        return text.count('\n') + 1

    df['calculated_verses'] = df['poem_text'].apply(count_verses)

    # Identify and report mismatches
    mismatches = df[df['calculated_verses'] != df['poem_verses']]
    print(f"Number of rows with mismatched verse count: {len(mismatches)}")

    # Discard mismatched rows
    df = df[df['calculated_verses'] == df['poem_verses']].drop(columns=['calculated_verses'])

    return df


## Process Arabic Poetry Melody

In [ ]:
datasets['Arabic-Poetry-Melody']

,poem_id,poem,verses,era,emotion
0,1,أَيُنعَى قتيلٌ قد قَضَى مستشهداً أًيٌبكًي شهيد...,59,العصر العثماني,sad
1,2,لِمن مربع بالسفحِ أَقوت ملاعبه وأصبحَ في مغناه...,40,العصر العثماني,sad
2,3,يا مَربعاً أخنا عليهِ بلاؤه بعدَ الخليطِ وطالَ...,26,العصر العثماني,sad
3,4,ستبدي خفيّات الأمورِ العواقبُ وَيظهرُ مِن سرّ ...,37,العصر العثماني,sad
4,5,أَلا غنّيا بالعامرية واِطربِ وَهاتِ لنا شرح ال...,41,العصر العثماني,sad
...,...,...,...,...,...
9447,9448,على أبن العصب الملح ي يثني اليوم من أثنى على ا...,38,العصر العباسي,joy
9448,9449,إنّ الأميرَ المُعلَّى في مَعالِيه أَدَقَّ حَظّ...,4,العصر المملوكي,joy
9449,9450,يا حسن دير سعيد إذ مررت به والأرض بالزهر في وش...,7,العصر العباسي,joy
9450,9451,وعارض مثل داره البدر دار بوجه كليلة القدر فلو ...,2,العصر العباسي,joy


In [ ]:
print(datasets['Arabic-Poetry-Melody']['poem'].iloc[2])

يا مَربعاً أخنا عليهِ بلاؤه بعدَ الخليطِ وطالَ فيه ثواؤهُ بانَت أَوانس خنسهِ وظبائهِ فتكنّسته خنسهُ وظباؤهُ أَنعم سلاماً من كئيبٍ مدنفٍ مَنَعته عَن سلوانهِ أهواؤهُ كلفٌ نَأى الأحباب عنه فَأسبلت عَبراته وتقطّعت أحشاؤهُ متصعّدُ الأنفاسِ راحةُ قلبهِ وَصلُ الأحبّةِ والفراق شقاؤهُ يَعلو حنيناً كلّما حنّت له تحتَ الرحالِ وأَرزمت أَنضاؤهُ وَأغنّ مَعسول المَراشف أغيد يُحيي وَيتلف لطفه وحباؤهُ قبّلت وردةَ خدّه فتضرّجت خجلاً وَحال على الخدودِ حياؤهُ شغفَ الفؤادُ بهِ وهامَ لأنّه دونَ الأنامِ سقامُه وشفاؤهُ فَلحاظُ أَعيُنه الفواتر داؤهُ وَرضابُ مبسمهِ اللذيذ دواؤهُ وَكأنّما زهر الإقاحةِ كشره وكأنّما زهر العبير شذاؤهُ نَشوان ييأس هجره وصدوده منه ويقطع مزحه ورضاؤهُ وَأحمّ منبجس المدامعِ نارُه في بطنه من حيث كان وماؤهُ بدرٌ يَضوء إذا تَبرقع وجهه فيشفُّ برقعه به وملاؤهُ هَطلُ الغزالةِ مرجحنّ لم تزل مُتواترات بالحيا أنواؤهُ يَغشى أقاليم البلاد ملثّه وتعمّ آفاق السماء صباؤهُ فكأنّ جودَ فلاح وبل ربابه لمّا تجلجل وبلُه وحياؤهُ مَلكٌ سَمت شَرفاً سماءُ فخارهِ حتّى سَمت فوقَ السماءِ سماؤهُ وفتى يقلُّ البحر ع

In [ ]:
# ---------- Dataset 7: Arabic-Poetry-Melody ----------
def unify_poetry_melody(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    emotion_map = {
        'sad' : 'حزن',
        'joy': 'فرح',
        'love': 'رومانسية',
    }

    # Map emotions to Arabic equivalents
    df['emotion'] = df['emotion'].map(emotion_map)
    df.rename(columns={
        'poem_id': 'poem_id',
        'poem': 'poem_text',
        'verses': 'poem_verses',
        'era': 'poet_era',
        'emotion': 'genre',
    }, inplace=True)
    return df



## Process Poems Hakim

In [ ]:
datasets['poems_hakim']['genre'].value_counts()

genre
قصيدة حزينه       1982
قصيدة دينية       1414
Empty             1355
قصيدة رومنسيه     1065
قصيدة رثاء        1024
قصيدة ذم           406
قصيدة المعلقات      10
قصيدة اعتذار         6
قصيدة الاناشيد       2
Name: count, dtype: int64

In [ ]:
# ---------- Dataset 8: poems_hakim ----------
def unify_poems_hakim(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()


    # Check if 'era' contains specific words, and assign to 'location' if not
    df['location'] = df['era'].apply(lambda x: x if "عصر" not in x and "المخضرمون" not in x else None)

    # Remove the moved values from the 'era' column
    df['era'] = df['era'].apply(lambda x: x if "عصر" in x or "المخضرمون" in x else None)
    df.rename(columns={
        'poem_name': 'poem_title',
        'poet_name': 'poet_name',
        'era': 'poet_era',
        'genre': 'genre',
        'poem': 'poem_text'
    }, inplace=True)

    # Remove the first line from 'poem_text' if it contains the word "صدر"
    df['poem_text'] = df['poem_text'].apply(lambda text: '\n'.join(text.split('\n')[1:]) if isinstance(text, str) and "صدر" in text else text)

    return df


## Process Boda Scrapped data

In [ ]:
datasets['boda_scrapped']

,title,poem_url,poet_page_url,id,poet,poem,tags,poem_no_diacritics,era,country,poem_title,poet_english_id,poet_description,source
0,Unknown,https://www.adab.com/post/view_post/0,NaN,poem_0.html,Unknown,NaN,[],NaN,NaN,NaN,Unknown,NaN,NaN,adab
1,العبودية الكبرى,https://www.adab.com/post/view_post/1,https://www.adab.com/Mustafa_Tal,poem_1.html,مصطفى وهبي التل (عرار),يا مدعي عام اللواء وخير من فهم القضية \nومناط ...,[],يا مدعي عام اللواء وخير من فهم القضية \nومناط ...,NaN,الأردن,العبودية الكبرى,Mustafa_Tal,مصطفى وهبي التل\r\n\r\nولد مصطفى وهبي بن صالح ...,adab
2,البكاء بين يدي زرقاء اليمامة,https://www.adab.com/post/view_post/10,https://www.adab.com/Amal_Donqol,poem_10.html,أمل دنقل,أيتها العرافة المقدَّسةْ ..\nجئتُ إليك .. مثخن...,[],أيتها العرافة المقدسة ..\nجئت إليك .. مثخنا با...,NaN,مصر,البكاء بين يدي زرقاء اليمامة,Amal_Donqol,"ولد في عام 1940 بقرية ""القلعة"", مركز ""قفط"" على...",adab
3,عجبت لحادينا المقحم سيره,https://www.adab.com/post/view_post/100,https://www.adab.com/Alfarazdaq,poem_100.html,الفرزدق,عَجِبْتُ لحادِينا المُقَحِّمِ سَيْرُهُ\nبِنا م...,[],عجبت لحادينا المقحم سيره\nبنا مزحفات من كلال و...,NaN,NaN,عجبت لحادينا المقحم سيره,Alfarazdaq,الفَرَزدَق\r\n38 - 110 هـ / 658 - 728 م\r\nهما...,adab
4,جزر الملح,https://www.adab.com/post/view_post/1000,https://www.adab.com/Muthaffar_Anawab,poem_1000.html,مظفر النواب,الآن ....\nوالعلم برتقالة\nتدور في بنفسج الأرو...,[],الآن ....\nوالعلم برتقالة\nتدور في بنفسج الأرو...,NaN,العراق,جزر الملح,Muthaffar_Anawab,مظفر النواب شاعر عربي واسع الشهرة ، عرفته عواص...,adab
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
444500,‏بلا عنوان - تيسير سبول - الديوان,https://www.aldiwan.net/poem99995.html,https://www.aldiwan.net/cat-poet-Tayseer-Sebol,poem_99995.html,تيسير سبول,(1)\nأنا يا صديقي\n‏أسير مع الوهم أدري\nأيمّم ...,"['قصائد عامه', 'نثريه']",(1)\nأنا يا صديقي\n‏أسير مع الوهم أدري\nأيمم ....,الأردن,NaN,‏بلا عنوان,poet-Tayseer-Sebol,تيسير رزق عبدالرحمن سبول.ولد في 15 يناير 1939م...,diwan
444501,شتاء لا يرحل - تيسير سبول - الديوان,https://www.aldiwan.net/poem99996.html,https://www.aldiwan.net/cat-poet-Tayseer-Sebol,poem_99996.html,تيسير سبول,على أفقنا تتمطى الغيوم\nتجوب ببط ء تخوم السماء...,"['قصائد عامه', 'نثريه']",على أفقنا تتمطى الغيوم\nتجوب ببط ء تخوم السماء...,الأردن,NaN,شتاء لا يرحل,poet-Tayseer-Sebol,تيسير رزق عبدالرحمن سبول.ولد في 15 يناير 1939م...,diwan
444502,قصيدة الضد - حيدر محمود - الديوان,https://www.aldiwan.net/poem99997.html,https://www.aldiwan.net/cat-poet-Haider-Mahmoud,poem_99997.html,حيدر محمود,قُضِيَ الأمرُ،\nوانتهى كلُّ شيءٍ..\nفوداعاً..\...,"['قصائد عامه', 'نثريه']",قضي الأمر،\nوانتهى كل شيء..\nفوداعا..\nيا كل ش...,الأردن,NaN,قصيدة الضد,poet-Haider-Mahmoud,حيدر محمود هو شاعرأردني من أصول فلسطينية ولد ف...,diwan
444503,الآخر - حيدر محمود - الديوان,https://www.aldiwan.net/poem99998.html,https://www.aldiwan.net/cat-poet-Haider-Mahmoud,poem_99998.html,حيدر محمود,(إلى مؤنس الرزاز)\nأحياناً يكرهني حدَّ الموتِ....,"['قصائد عامه', 'نثريه']",(إلى مؤنس الرزاز)\nأحيانا يكرهني حد الموت..\nو...,الأردن,NaN,الآخر,poet-Haider-Mahmoud,حيدر محمود هو شاعرأردني من أصول فلسطينية ولد ف...,diwan


In [ ]:
datasets['boda_scrapped'].columns

Index(['title', 'poem_url', 'poet_page_url', 'id', 'poet', 'poem', 'tags',
       'poem_no_diacritics', 'era', 'country', 'poem_title', 'poet_english_id',
       'poet_description', 'source'],
      dtype='object')

In [ ]:


def process_poem_text(df: pd.DataFrame) -> pd.DataFrame:
    """
    Processes the 'poem' column in the DataFrame based on the source.
    If the source is 'adab', it removes multiple new lines and replaces them with a single one,
    and removes tab characters.
    """
    df = df.copy()


    def clean_text(text, source):
        if source == "adab" and isinstance(text, str):
            text = re.sub(r'\n+', '\n', text)  # Replace multiple new lines with a single one
            text = text.replace('\t', '')  # Remove tab characters


        if source == "poets_gate" and isinstance(text, str):
            lines = text.split('\n')
            if len(lines) % 2 == 0:
                verses = []
                for i in range(0, len(lines), 2):
                    verses.append(f"{lines[i]}\t{lines[i+1]}")
                text = '\n'.join(verses)
            else:
                text = '\n'.join(lines)
        return text

    df['poem'] = df.apply(lambda row: clean_text(row['poem'], row['source']), axis=1)
    
    return df


def clean_era_and_extract_location(df):
    # List of Arab countries to extract as location
    arab_countries = [
        'الإمارات', 'السعودية', 'البحرين', 'قطر', 'الكويت', 'عمان', 'اليمن', 'مصر', 'الأردن',
        'فلسطين', 'العراق', 'سوريا', 'لبنان', 'تونس', 'الجزائر', 'المغرب', 'ليبيا',
        'السودان', 'موريتانيا', 'السنغال', 'الصومال'
    ]
    # Entries to drop
    bad_era_values = {
        'Unknown', '>', 'أشعار موضوعية', 'الشعراء الأعضاء .. عامِّي', 'ذكاء اصطناعي'
    }
    # Move countries to a new location column
    df['country'] = df['era'].where(df['era'].isin(arab_countries), None)
    # Remove rows with unwanted era values or "ذكاء اصطناعي"
    df = df[~df['era'].isin(bad_era_values)]
    # Optionally, replace country values in `era` with NaN since they're now in `location`
    df['era'] = df['era'].where(~df['era'].isin(arab_countries), None)
    # Reset index for cleanliness
    return df.reset_index(drop=True)


def extract_tags_metadata(df: pd.DataFrame) -> pd.DataFrame:
    """
    Processes the 'tags' column in the DataFrame and extracts relevant metadata
    into the columns: 'genre', 'poem_type', 'rhyme', 'meter'.
    """
    df = df.copy()
    
    # Initialize target columns
    for col in ['genre', 'poem_type', 'rhyme', 'meter']:
        df[col] = None

    def process_tags(tags):
        genre, poem_type, rhyme, meter = None, None, None, None
        if not isinstance(tags, list):
            return genre, poem_type, rhyme, meter

        for tag in tags:
            tag = tag.strip()
            if 'قصائد' in tag:
                genre = tag
            elif any(x in tag for x in ['عموديه', 'التفعيله', 'نثريه']):
                poem_type = tag
            elif 'قافية' in tag:
                rhyme = tag
            elif 'بحر' in tag:
                meter = tag
        return genre, poem_type, rhyme, meter

    df[['genre', 'poem_type', 'rhyme', 'meter']] = df['tags'].apply(lambda tags: pd.Series(process_tags(tags)))
    
    return df
def clean_title(df: pd.DataFrame) -> pd.DataFrame:
    """
    Cleans the title column by splitting on '-' and keeping the first part
    only for rows where source is 'diwan'.
    """
    df = df.copy()
    is_diwan = df['source'] == 'diwan'
    df.loc[is_diwan, 'title'] = df.loc[is_diwan, 'title'].astype(str).str.split('-').str[0].str.strip()
    return df

def unify_boda_scrapped(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()


    # Clean title
    df = clean_title(df)
    # Process poem text
    df = process_poem_text(df)
    # Extract tags metadata
    df = extract_tags_metadata(df)
    
    df = clean_era_and_extract_location(df)

    df.dropna(subset=['poem'], inplace=True)

    df.drop(columns=['title'], inplace=True)

    df.drop(columns=['poem_no_diacritics'], inplace=True)

    # Rename columns
    df.rename(columns={
        # 'title': 'poem_title',
        'id' : 'poem_id',
        'poem': 'poem_text',
        'country': 'location',
        'poet': 'poet_name',
        'era': 'poet_era',
        'poet_page_url': 'poet_url',
    }, inplace=True)
    
    return df





## Process Rania Data

In [ ]:
datasets['mawsooaa']


,poet_name,age_period,age_id,meter,rhyme_letter,poetry_type,verse_count,poem_title,hemistichs_format,theme,poem_id,poet_gender,poem_length,word_count,source_file
0,عَمرُو بنُ قَمِيئَة,قبل الإسلام,1,الطويل,د,فصيح,11,خَلِيلَيَّ لَا تَسْتَعْجِلَا أَنْ تَزَوَّدا,وَأَنْ تَجْمَعــا شــَمْلِي وَتَنْتَظِـرا غَـد...,غير محدد,1,ذكر,1109,115,complete_age_1_قبل_الإسلام.json
1,عَمرُو بنُ قَمِيئَة,قبل الإسلام,1,الطويل,ح,فصيح,28,أَرَى جَارَتِي خَفَّتْ وَخَفَّ نَصِيحُها,وَحُــبَّ بِهـا لَـوْلا النَّـوَى وَطُمُوحُهـا...,غير محدد,2,ذكر,2702,268,complete_age_1_قبل_الإسلام.json
2,عَمرُو بنُ قَمِيئَة,قبل الإسلام,1,الطويل,م,فصيح,15,إِنْ أَكُ قَدْ أَقْصَرْتُ عَنْ طُولِ رِحْلَةٍ,فَيَـــا رُبَّ أَصــْحَابٍ بَعَثْــتُ كِــرَام...,غير محدد,3,ذكر,1453,155,complete_age_1_قبل_الإسلام.json
3,عَمرُو بنُ قَمِيئَة,قبل الإسلام,1,المنسرح,م,فصيح,6,يَا لَهْفَ نَفْسِي عَلَى الشَّبَابِ وَلَمْ,أَفْقِــدْ بِـهِ إِذْ فَقَـدْتُهُ أَمَمـا\t\tي...,غير محدد,4,ذكر,501,61,complete_age_1_قبل_الإسلام.json
4,عَمرُو بنُ قَمِيئَة,قبل الإسلام,1,المتقارب,ل,فصيح,13,تَحِنُّ حَنِيناً إِلَى مَالِكٍ,فَحِنِّــي حَنِينَــكِ إِنِّــي مُعَـالِي\t\tت...,غير محدد,5,ذكر,1100,109,complete_age_1_قبل_الإسلام.json
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20209,ابن معتوق الموسوي,العصر العثماني,11,الكامل,ء,فصيح,76,هذا الحِمى فاِنزِلْ على جَرعائِهِ,واِحْـذَرْ ظُبـا لَفَتـاتِ عِيـنِ ظِبـائِهِ\t\...,"الحرب, الحب, الفخر, الرثاء, الطبيعة, الدين, ال...",41915,ذكر,5701,747,complete_age_11_العصر_العثماني_reconstructed.json
20210,ابن معتوق الموسوي,العصر العثماني,11,الكامل,ب,فصيح,72,مِيلوا بنا نحوَ الحجونِ ونكّبوا,حيــثُ الهــوى منـه فثـمّ المَطلَـبُ\t\tمِيلـو...,"الحرب, الحب, الرثاء, الطبيعة, القبيلة, الحكمة",41916,ذكر,5517,703,complete_age_11_العصر_العثماني_reconstructed.json
20211,ابن معتوق الموسوي,العصر العثماني,11,الكامل,ر,فصيح,73,كتمَ الهوى فوشى النّحولُ بسرِّه,وصــحا فحيّــاهُ النســيمُ بخمـرِهِ\t\tكتـمَ ا...,"الحرب, الحب, الفخر, الرثاء, الطبيعة, الدين, ال...",41917,ذكر,5585,715,complete_age_11_العصر_العثماني_reconstructed.json
20212,ابن معتوق الموسوي,العصر العثماني,11,الكامل,ن,فصيح,71,ضربوا القِبابَ وطنّبوها بالقَنا,فمَحَـوا بأنجُمِهـا مصـابيحَ المُنـا\t\tضـربوا...,"الحرب, الحب, الفخر, الدين, القبيلة, الغربة, ال...",41918,ذكر,5424,699,complete_age_11_العصر_العثماني_reconstructed.json


In [ ]:
datasets['arapoet']

,title,tags,verse_count,author,era,meter,genre,hemistichs_format
0,حَيّاكُمُ اللَهُ أَحيوا العِلمَ وَالأَدَبا,['البسيط' 'سياسية' 'العصر الحديث'],40,حافظ ابراهيم,العصر الحديث,البسيط,سياسية,حَيّـاكُمُ اللَـهُ أَحيـوا العِلمَ وَالأَدَبا\...
1,غابَ الأَديبُ أَديبُ مِصرٍ وَاِختَفى,['الكامل' 'رثاء' 'العصر الحديث'],3,حافظ ابراهيم,العصر الحديث,الكامل,رثاء,غـابَ الأَديـبُ أَديـبُ مِصرٍ وَاِختَفى\t\tفَل...
2,عُثمانُ إِنَّكَ قَد أَتَيتَ مُوَفَّقاً,['الكامل' 'مدح' 'العصر الحديث'],3,حافظ ابراهيم,العصر الحديث,الكامل,مدح,عُثمـانُ إِنَّكَ قَد أَتَيتَ مُوَفَّقاً\t\tشـَ...
3,إِنَّ عَضّيكَ يا أَخي بِالمَلامِ,['الخفيف' 'عتاب' 'العصر الحديث'],10,حافظ ابراهيم,العصر الحديث,الخفيف,عتاب,إِنَّ عَضــّيكَ يــا أَخــي بِــالمَلامِ\t\tلا...
4,مِن واجِدٍ مُنَقِّرِ المَنامِ,['الرجز' 'عتاب' 'العصر الحديث'],15,حافظ ابراهيم,العصر الحديث,الرجز,عتاب,مِن واجِدٍ مُنَقِّرِ المَنامِ\t\tطَريدَ دَهرٍ ...
...,...,...,...,...,...,...,...,...
9408,إِذا ما أَلحَدَت أُمَمٌ بِجَهلٍ,"['الوافر', 'هجاء', 'قبل الإسلام']",3,أَبو العَلاء المَعَرِي,قبل الإسلام,الوافر,هجاء,إِذا مـا أَلحَدَت أُمَمٌ\t\tبِجَهلٍ فَقابِلهـا...
9409,تَلا كِتابَ اللَهِ مِن حِفظِهِ,"['هجاء', 'قبل الإسلام']",3,أَبو العَلاء المَعَرِي,قبل الإسلام,NaN,هجاء,تَلا كِتـابَ اللَهِ مِن حِفظِهِ\t\tمَن هُو بِا...
9410,كَأَنَّما دُنياكَ وَحشِيَّةٌ,"['هجاء', 'قبل الإسلام']",7,أَبو العَلاء المَعَرِي,قبل الإسلام,NaN,هجاء,كَأَنَّمــا دُنيــاكَ وَحشــِيَّةٌ\t\tنَظَـرَت...
9411,زَعَمَ الزاعِمونَ وَالقَولُ مِن مَي,"['الخفيف', 'هجاء', 'قبل الإسلام']",2,أَبو العَلاء المَعَرِي,قبل الإسلام,الخفيف,هجاء,زَعَمَ الزاعِمونَ وَالقَولُ مِن مَي\t\tنٍ وَصـ...


In [ ]:
datasets['adab_world']

,poet_name,poet_era,poet_era_key,poet_era_period,poet_birth_death,page,title,url,full_url,post_id,image_url,content_scraped,explanation,vocabulary,meter,theme,verses.1,verses_formatted
0,امرؤ القيس,العصر الجاهلي,pre_islamic,400 ~ 610 ميلادية,(501 – 540) ميلادية,1,ألا انعم صباحا أيها الربع وانطق,https://adabworld.com/%d9%82%d8%b5%d9%8a%d8%af...,https://adabworld.com/%d9%82%d8%b5%d9%8a%d8%af...,96983,NaN,True,NaN,NaN,NaN,NaN,أَلا اِنعِم صَباحاً أَيُّها الرَبعُ وَاِنطِقِ\...,أَلا اِنعِم صَباحاً أَيُّها الرَبعُ وَاِنطِقِ\...
1,امرؤ القيس,العصر الجاهلي,pre_islamic,400 ~ 610 ميلادية,(501 – 540) ميلادية,1,ألا يا لهف هند إثر قوم,https://adabworld.com/%d9%82%d8%b5%d9%8a%d8%af...,https://adabworld.com/%d9%82%d8%b5%d9%8a%d8%af...,96996,NaN,True,NaN,NaN,NaN,NaN,أَلا يا لَهفَ هِندٍ إِثرَ قَومٍ\tهُمُ كانوا ال...,أَلا يا لَهفَ هِندٍ إِثرَ قَومٍ\tهُمُ كانوا ال...
2,امرؤ القيس,العصر الجاهلي,pre_islamic,400 ~ 610 ميلادية,(501 – 540) ميلادية,1,أرانا موضعين لأمر غيب,https://adabworld.com/%d9%82%d8%b5%d9%8a%d8%af...,https://adabworld.com/%d9%82%d8%b5%d9%8a%d8%af...,96999,NaN,True,NaN,NaN,NaN,NaN,أَرانا موضِعينَ لِأَمرِ غَيبٍ\tوَنُسحَرُ بِالط...,أَرانا موضِعينَ لِأَمرِ غَيبٍ\tوَنُسحَرُ بِالط...
3,امرؤ القيس,العصر الجاهلي,pre_islamic,400 ~ 610 ميلادية,(501 – 540) ميلادية,1,رب رام من بني ثعل,https://adabworld.com/%d9%82%d8%b5%d9%8a%d8%af...,https://adabworld.com/%d9%82%d8%b5%d9%8a%d8%af...,96986,NaN,True,NaN,NaN,NaN,NaN,رُبَّ رامٍ مِن بَني ثُعَلٍ\tمُتلِجٍ كَفَّيهِ ف...,رُبَّ رامٍ مِن بَني ثُعَلٍ\tمُتلِجٍ كَفَّيهِ ف...
4,امرؤ القيس,العصر الجاهلي,pre_islamic,400 ~ 610 ميلادية,(501 – 540) ميلادية,1,يا دار ماوِية بالحائلِ,https://adabworld.com/%d9%82%d8%b5%d9%8a%d8%af...,https://adabworld.com/%d9%82%d8%b5%d9%8a%d8%af...,96977,NaN,True,NaN,NaN,NaN,NaN,يا دارَ ماوِيَّةَ بِالحائِلِ\tفَالسُهبِ فَالخَ...,يا دارَ ماوِيَّةَ بِالحائِلِ\tفَالسُهبِ فَالخَ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2017,ابن معصوم,العصر العثماني,ottoman,1517 ~ 1798 ميلادية,(1642 – 1708 ميلادية),34,إليك فقلبي لا تقر بلابله,https://adabworld.com/%d9%82%d8%b5%d9%8a%d8%af...,https://adabworld.com/%d9%82%d8%b5%d9%8a%d8%af...,116365,NaN,True,NaN,NaN,NaN,NaN,إِليكَ فَقَلبي لا تقِرُّ بلابلُه\tإِذا ما شدَت...,إِليكَ فَقَلبي لا تقِرُّ بلابلُه\tإِذا ما شدَت...
2018,سعيد بن أحمد البوسعيدي,العصر العثماني,ottoman,1517 ~ 1798 ميلادية,(#- 1811 ميلادية),1,لهفي على عيش مضى,https://adabworld.com/%d9%82%d8%b5%d9%8a%d8%af...,https://adabworld.com/%d9%82%d8%b5%d9%8a%d8%af...,141357,https://adabworld.com/wp-content/uploads/2022/...,True,NaN,NaN,NaN,NaN,لهفي على عيش مضى\tما ذقت أحلى منه شي\nلما ذكرت...,لهفي على عيش مضى\tما ذقت أحلى منه شي\nلما ذكرت...
2019,سعيد بن أحمد البوسعيدي,العصر العثماني,ottoman,1517 ~ 1798 ميلادية,(#- 1811 ميلادية),1,وافا حمامك يا حبيبي بالعجل,https://adabworld.com/%d9%82%d8%b5%d9%8a%d8%af...,https://adabworld.com/%d9%82%d8%b5%d9%8a%d8%af...,141359,https://adabworld.com/wp-content/uploads/2022/...,True,NaN,NaN,NaN,NaN,وافا حمامك يا حبيبي بالعجل\tنار تلهب في ضميري ...,وافا حمامك يا حبيبي بالعجل\tنار تلهب في ضميري ...
2020,سعيد بن أحمد البوسعيدي,العصر العثماني,ottoman,1517 ~ 1798 ميلادية,(#- 1811 ميلادية),1,إذا شحت الخضراء بالويل فالتمس,https://adabworld.com/%d9%82%d8%b5%d9%8a%d8%af...,https://adabworld.com/%d9%82%d8%b5%d9%8a%d8%af...,141360,https://adabworld.com/wp-content/uploads/2022/...,True,NaN,NaN,NaN,NaN,إذا شحت الخضراء بالويل فالتمس\tتجد جود سلطان ع...,إذا شحت الخضراء بالويل فالتمس\tتجد جود سلطان ع...


In [ ]:
def unify_adab_world(df):

    df = df.copy()
    df['poet_name'] = df['poet_name'].str.strip()
    df['poet_era'] = df['poet_era'].str.strip()
    df['verses_formatted'] = df['verses_formatted'].str.replace('\t\t', '\t', regex=False)

    delete_columns = [ 'poet_era_key','poet_era_period','poet_birth_death','page','url','image_url', 'verses.1', 'content_scraped','vocabulary','explanation', 'post_id','theme']
    rename_map = {
        'title': 'poem_title',
        'verses_formatted': 'poem_text',
        'full_url': 'poem_url',
    }
    df.drop(columns=delete_columns, inplace=True, errors='ignore')
    df.rename(columns=rename_map, inplace=True)

    return df


def unify_arapoet(df):
    df = df.copy()



    def process_meter(meter):
        if meter == "عموديه":
            return meter  # Return the original meter value for "عموديه"
        if not meter.startswith("بحر"):
            meter = f"بحر {meter}"
        return meter
    df['meter'] = df['meter'].astype(str).apply(process_meter)
    df['poem_type'] = df['meter'].apply(lambda meter: "عموديه" if meter == "عموديه" else None)
    df['hemistichs_format'] = df['hemistichs_format'].str.replace('\t\t', '\t', regex=False)

    rename_map = {
        'title' : 'poem_title',
        'author': 'poet_name',
        'poet era': 'poet_era',
        'hemistichs_format': 'poem_text',}    
    df.rename(columns=rename_map, inplace=True)
    df.drop(columns=['verse_count',"explanation"], inplace=True, errors='ignore')
    
    return df



def unify_mawsooaa(df):
    df = df.copy()
    meter_values = ['مجزوء الرجز' , 'مجزوء الوافر', 'أحذ الكامل', 'مجزوء الوافر' ,'منهوك المنسرح', 'مشطور الرجز' ,'مجزوء الكامل المرفّل', 'منهوك الرجز']
    
    def adjust_meter_and_theme(row):
        if row['theme'] in meter_values:
            row['meter'] = f"بحر {row['theme']}"
            row['theme'] = None
        return row
    
    df['hemistichs_format'] = df['hemistichs_format'].str.replace('\t\t', '\t', regex=False)

    df = df.apply(adjust_meter_and_theme, axis=1)
    rename_map = {
        "age_period": "poet_era",
        "rhyme_letter": "rhyme",
        "poetry_type": "poem_language",
        'theme' : 'genre',
        'hemistichs_format': 'poem_text',
        'source_file': 'source',

    }

    remove_columns = ['age_period', 'age_id','verse_count','poet_gender','poem_length','word_count','poem_id']
    df.drop(columns=remove_columns, inplace=True, errors='ignore')
    df.rename(columns=rename_map, inplace=True)

    return df
    

# Apply Column name normalizaiton

In [ ]:
for key, dataset in datasets.items():
    if isinstance(dataset, pd.DataFrame):
        print(f"Dataset: {key}")
        print("Columns:", dataset.columns.tolist())
        print()
    elif isinstance(dataset, dict):
        print(f"Dataset: {key}")
        print("Keys:", list(dataset.keys()))
        print()
        

Dataset: FannOrFlop
Columns: ['source', 'title', 'tags', 'verse_count', 'author', 'era', 'meter', 'genre', 'id', 'explanation', 'poem_verses', 'raw_explanation']

Dataset: Arabic Poetry Dataset
Columns: ['poet_name', 'poet_era', 'poem_tags', 'poem_title', 'poem_text', 'poem_count']

Dataset: Arabic PCD
Columns: ['العصر', 'الشاعر', 'الديوان', 'القافية', 'البحر', 'الشطر الايسر', 'الشطر الايمن', 'البيت']

Dataset: AraPoems
Columns: ['poem_title', 'first_hemistich', 'second_hemistich', 'poet', 'meter', 'sub_meter', 'البحر', 'جزء البحر', 'era', 'rhyme', 'قافية', 'type_en', 'type_ar', 'link', 'gender', 'poem_text']

Dataset: Ashaar
Columns: ['poem title', 'poem meter', 'poem verses', 'poem theme', 'poem url', 'poet name', 'poet description', 'poet url', 'poet era', 'poet location', 'poem description', 'poem language type']

Dataset: Arabic Poetry Dataset (6th - 21st century)
Columns: ['poem_id', 'poem_link', 'poem_style', 'poem_text', 'poem_title', 'poet_cat', 'poet_id', 'poet_link', 'poet_n

In [ ]:

# Unify all datasets into a single DataFrame
unified_datasets = []

# Iterate through datasets and apply the appropriate unification function
for key, dataset in datasets.items():
    if isinstance(dataset, pd.DataFrame):
        if key == 'Arabic PCD':
            df = unify_arabic_pcd(dataset)
        elif key == 'AraPoems':
            df = unify_arapoems(dataset)
        elif key == 'FannOrFlop':
            df = unify_fann_or_flop(dataset)
        elif key == 'Arabic Poetry Dataset':
            df = unify_arabic_poetry_dataset(dataset)
        elif key == 'Ashaar':
            df = unify_ashaar(dataset)
        elif key == 'Arabic Poetry Dataset (6th - 21st century)':
            df = unify_century_dataset(dataset)
        elif key == 'Arabic-Poetry-Melody':
            df = unify_poetry_melody(dataset)
        elif key == 'poems_hakim':
            df = unify_poems_hakim(dataset)
        elif key == 'boda_scrapped':
            df = unify_boda_scrapped(dataset)
        elif key == 'mawsooaa':
            df = unify_mawsooaa(dataset)
        elif key == 'adab_world':
            df = unify_adab_world(dataset)
        elif key == 'arapoet':
            df = unify_arapoet(dataset)
        else:
            print(f"Skipping unrecognized dataset: {key}")
            continue  # Skip unknown keys

        # Add the dataset name as a column
        df['dataset_name'] = key

        # Append the modified DataFrame
        unified_datasets.append(df)



6917 valid rows
67 invalid rows
Number of rows with invalid or missing poem_tags: 2423
Examples of failed tags:
['بحر الطويل', 'عموديه, بحر الطويل,  قافية الدال (د) ', 'بحر الكامل', 'بحر الوافر', 'عموديه, بحر السريع,  قافية الميم (م) ', 'بحر البسيط', 'بحر المنسرح', 'عموديه, بحر الوافر,  قافية الياء (ي) ', 'قصائد حزينه, عموديه, بحر الخفيف', 'قصائد عامه, عموديه, بحر الوافر']
Number of rows with mismatched verse count: 950


/tmp/ipykernel_2971615/3485617546.py:48: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['era'] = df['era'].where(~df['era'].isin(arab_countries), None)


In [ ]:
for  dataset in unified_datasets:
    if isinstance(dataset, pd.DataFrame):
        print(f"Dataset: {dataset['dataset_name'].iloc[0]}")
        print("Columns:", dataset.columns.tolist())
        print()
    elif isinstance(dataset, dict):
        print(f"Dataset: {key}")
        print("Keys:", list(dataset.keys()))
        print()
        

Dataset: FannOrFlop
Columns: ['source', 'poem_title', 'tags', 'poet_name', 'poet_era', 'meter', 'genre', 'poem_id', 'overall_explanation', 'poem_text', 'verses_explanation', 'dataset_name']

Dataset: Arabic Poetry Dataset
Columns: ['poet_name', 'poet_era', 'tags', 'poem_title', 'poem_text', 'poem_verses', 'genre', 'poem_type', 'meter', 'rhyme', 'dataset_name']

Dataset: Arabic PCD
Columns: ['poet_era', 'poet_name', 'rhyme', 'meter', 'poem_text', 'dataset_name']

Dataset: AraPoems
Columns: ['poem_title', 'poet_name', 'poet_era', 'genre', 'source', 'poem_text', 'dataset_name']

Dataset: Ashaar
Columns: ['poem_title', 'meter', 'poem_text', 'genre', 'poem_url', 'poet_name', 'poet_description', 'poet_url', 'poet_era', 'overall_explanation', 'dataset_name']

Dataset: Arabic Poetry Dataset (6th - 21st century)
Columns: ['poem_id', 'source', 'poem_text', 'poem_title', 'poet_id', 'poet_url', 'poet_name', 'poet_era', 'location', 'dataset_name']

Dataset: Arabic-Poetry-Melody
Columns: ['poem_id',

In [ ]:
# Combine and drop all existing index info

# Ensure all columns are of type str and gather all unique column names
all_columns = set()
for df in unified_datasets:
    if isinstance(df, pd.DataFrame):
        df.columns = df.columns.astype(str)
        all_columns.update(df.columns)
# Add missing columns with NaN to each dataset
for i, df in enumerate(unified_datasets):
    if isinstance(df, pd.DataFrame):
        missing_columns = all_columns - set(df.columns)
        for col in missing_columns:
            df[col] = pd.NA
        unified_datasets[i] = df[sorted(all_columns)]  # Sort columns for consistency

# cleaned_dfs = []

# for i, df in enumerate(unified_datasets):
#     df_clean = df.copy()

#     # Ensure columns are unique
#     assert df_clean.columns.is_unique, f"Dataset {i} has non-unique columns"

#     # Ensure index is not a MultiIndex and is unique
#     df_clean.index = pd.RangeIndex(len(df_clean))  # Replace index completely

#     cleaned_dfs.append(df_clean)

# Now try the concatenation
all_poems_df = pd.concat(unified_datasets, ignore_index=True)



# Replace "Empty" and NaN values with None
all_poems_df.replace("Empty", None, inplace=True)
# all_poems_df.fillna(value=None, inplace=True)


# Populate 'poem_verses' column if it is empty or NaN
all_poems_df['poem_verses'] = all_poems_df['poem_verses'].fillna(
    all_poems_df['poem_text'].apply(lambda x: x.count('\n') + 1 if isinstance(x, str) else None)
)

# Ensure unique values for the 'poem_id' column
if 'poem_id' in all_poems_df.columns:
    all_poems_df['poem_id'] = all_poems_df.index + 1



/tmp/ipykernel_2971615/820768149.py:31: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_poems_df = pd.concat(unified_datasets, ignore_index=True)


In [ ]:
# Count empty, NaN, None, or 'Empty' values **per column**
empty_nan_none_counts = {
    column: (
        all_poems_df[column].apply(
            lambda x: pd.isna(x) or x == '' or x == 'Empty'
        ).sum()
    )
    for column in all_poems_df.columns
}

# Display the counts
for col, count in empty_nan_none_counts.items():
    print(f"{col}: {count}")


dataset_name: 0
era: 2821195
genre: 2544221
location: 2765839
meter: 830079
overall_explanation: 2823205
poem_id: 0
poem_language: 2809908
poem_text: 127
poem_title: 1933021
poem_type: 2759402
poem_url: 2181658
poem_verses: 44
poet_description: 2503766
poet_english_id: 2438232
poet_era: 289378
poet_id: 2772101
poet_name: 9943
poet_url: 2303045
rhyme: 917178
source: 2187571
tags: 2349357
verses_explanation: 2823205


In [ ]:
import re

def remove_diacritics(text):
    """Remove Arabic diacritics from the given text."""
    arabic_diacritics = re.compile(r'[\u064B-\u0652]')
    return arabic_diacritics.sub('', text)


# Remove rows with empty or NaN values in the 'poem_text' column
all_poems_df = all_poems_df.dropna(subset=['poem_text']).loc[all_poems_df['poem_text'].str.strip() != '']

# Create a new column with poem text without diacritics
all_poems_df['poem_text_no_diacritics'] = all_poems_df['poem_text'].apply(lambda x: remove_diacritics(x) if isinstance(x, str) else x)
# Remove diacritics from the 'poet_name' column
all_poems_df['poet_name_with_diacritics'] = all_poems_df['poet_name']
all_poems_df['poet_name'] = all_poems_df['poet_name'].apply(lambda x: remove_diacritics(x) if isinstance(x, str) else x)


# Filter rows belonging to the 'FannOrFlop' dataset
test_df = all_poems_df[all_poems_df['dataset_name'] == 'FannOrFlop']
train_df = all_poems_df[all_poems_df['dataset_name'] != 'FannOrFlop']


In [ ]:
# Count empty, NaN, None, or 'Empty' values **per column**
empty_nan_none_counts = {
    column: (
        all_poems_df[column].apply(
            lambda x: pd.isna(x) or x == '' or x == 'Empty'
        ).sum()
    )
    for column in all_poems_df.columns
}

# Display the counts
for col, count in empty_nan_none_counts.items():
    print(f"{col}: {count}")

dataset_name: 0
era: 2821060
genre: 2544172
location: 2765708
meter: 830030
overall_explanation: 2823070
poem_id: 0
poem_language: 2809783
poem_text: 0
poem_title: 1933019
poem_type: 2759343
poem_url: 2181563
poem_verses: 0
poet_description: 2503638
poet_english_id: 2438104
poet_era: 289364
poet_id: 2771968
poet_name: 9943
poet_url: 2302919
rhyme: 917129
source: 2187455
tags: 2349309
verses_explanation: 2823070
poem_text_no_diacritics: 0
poet_name_with_diacritics: 9943


# Post Processing

In [ ]:

print('Dedupe train data shape:', train_df.shape)
print('Dedupe test data shape:', test_df.shape)

Dedupe train data shape: (2823070, 25)
Dedupe test data shape: (6917, 25)


In [ ]:

# 1. Merge `gnre` into `genre` if genre is missing or empty
for df in [train_df, test_df]:
    df['genre'] = df['genre'].fillna('').astype(str).str.strip()

# 2. Strip leading/trailing spaces from all string columns
def strip_all_columns(df):
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = df[col].astype(str).str.strip()
    return df

train_df = strip_all_columns(train_df)
test_df = strip_all_columns(test_df)

# 3. Define mappings
country_names = [
    'مصر', 'لبنان', 'المغرب', 'سوريا', 'تونس', 'فلسطين', 'موريتانيا', 'العراق',
    'اليمن', 'ليبيا', 'السعودية', 'السودان', 'الجزائر', 'الأردن', 'عمان',
    'الإمارات',
]

era_mapping = {
    'الحديث': 'العصر الحديث',
    'العثماني': 'العصر العثماني',
    'المملوكي': 'العصر المملوكي',
    'العباسي': 'العصر العباسي',
    'الأموي': 'العصر الأموي',
    'الفاطمي': 'العصر الفاطمي',
    'الأيوبي': 'العصر الأيوبي',
    'قبل الإسلام': 'العصر الجاهلي',
    'العصر الايوبي': 'العصر الأيوبي',
    'العصر الاموي': 'العصر الأموي',
    'الدولة الايوبية': 'العصر الأيوبي',
    'الدولة الفاطمية': 'العصر الفاطمي',
    'الدولة المملوكية': 'العصر المملوكي',
    'بين الدولتين' : 'عصر بين الدولتين',
    'الإسلامي': 'العصر الإسلامي',
    'العصر الاسلامي': 'العصر الإسلامي',
    'قبل الإسلام': 'العصر الجاهلي',
    
    # Standardize all to المخضرمون
    'المخضرمون': 'المخضرمون',
    'المخضرمين': 'المخضرمون',
    'الشعراء المخضرمون': 'المخضرمون',
}


def normalize_poet_era_location_and_meter(df):
    for idx, row in df.iterrows():
        # Normalize poet_era and location
        era = row.get('poet_era', '')
        location = row.get('location', '')
        
        if era in country_names:
            if not location or location.strip() == '':
                df.at[idx, 'location'] = era
            elif era not in location:
                df.at[idx, 'location'] = era  # Overwrite to keep clean
            df.at[idx, 'poet_era'] = 'غير محدد'
        elif era in era_mapping:
            df.at[idx, 'poet_era'] = era_mapping[era]
        
        # Normalize meter
        meter = row.get('meter', '')
        if meter:
            # Remove surrounding whitespace
            meter = meter.strip()
            
            # If meter starts with "البحر" or does not start with "بحر"
            if meter.startswith("البحر"):
                normalized = re.sub(r"^البحر\s+", "", meter)
                df.at[idx, 'meter'] = f"بحر {normalized}"
            elif not meter.startswith("بحر"):
                df.at[idx, 'meter'] = f"بحر {meter}"



    # Set 'poet_era' to None if its value is in the specified list
    df['poet_era'] = df['poet_era'].apply(lambda x: None if x in ["عصرين", "غير محدد"] else x)
    
    # Normalize 'meter' column for specific cases
    df['meter'] = df['meter'].apply(lambda x: None if x in ["بحر nan", "بحر None", "بحر"] else x)
    return df


def detect_rhyme(poem_text):
    """
    Detect rhyme character from poem verses.
    If at least 70% of verses end with the same character, return that character as rhyme.
    """
    if pd.isna(poem_text) or not isinstance(poem_text, str):
        return None
    
    # Split poem into verses
    verses = [verse.strip() for verse in poem_text.split('\n') if verse.strip()]
    
    if len(verses) < 2:  # Need at least 2 verses to determine rhyme
        return None
    
    # Count ending characters
    ending_chars = {}
    for verse in verses:
        if verse:  # Make sure verse is not empty
            last_char = verse[-1]
            ending_chars[last_char] = ending_chars.get(last_char, 0) + 1
    
    # Find the most common ending character
    if not ending_chars:
        return None
    
    max_count = max(ending_chars.values())
    total_verses = len(verses)
    
    # Check if at least 70% of verses end with the same character
    if max_count / total_verses >= 0.7:
        # Return the character that appears most frequently
        for char, count in ending_chars.items():
            if count == max_count:
                return char
    
    return None


for df in [train_df, test_df]:
    mask = df['rhyme'].isna() | (df['rhyme'] == '')
    df.loc[mask, 'rhyme'] = df.loc[mask, 'poem_text'].apply(detect_rhyme)

# Apply to both splits
train_df = normalize_poet_era_location_and_meter(train_df)
test_df = normalize_poet_era_location_and_meter(test_df)



/tmp/ipykernel_2971615/3903920934.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['genre'] = df['genre'].fillna('').astype(str).str.strip()
/tmp/ipykernel_2971615/3903920934.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['genre'] = df['genre'].fillna('').astype(str).str.strip()
/tmp/ipykernel_2971615/3903920934.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the docume

In [ ]:
# train_df.to_csv('../data/cleaned_train.tsv', sep='\t', index=False)
test_df.to_csv('/path/to/data/test_data_v2.tsv', sep='\t', index=False)

print("✅ Cleaned data saved to data/cleaned_train.tsv and data/cleaned_test.tsv")

✅ Cleaned data saved to data/cleaned_train.tsv and data/cleaned_test.tsv


In [ ]:

total_count = 0
for key, dataset in datasets.items():
    if isinstance(dataset, pd.DataFrame):
        count = len(dataset)
        print(f"{key}: {count} rows")
        total_count += count
    elif isinstance(dataset, dict):
        count = len(dataset)
        print(f"{key}: {count} entries")
        total_count += count

print(f"Total: {total_count}")

train_df.columns.tolist()

FannOrFlop: 6984 rows
Arabic Poetry Dataset: 75022 rows
Arabic PCD: 1831770 rows
AraPoems: 165509 rows
Ashaar: 254590 rows
Arabic Poetry Dataset (6th - 21st century): 58021 rows
Arabic-Poetry-Melody: 9452 rows
poems_hakim: 7264 rows
boda_scrapped: 444505 rows
adab_world: 2022 rows
arapoet: 9413 rows
mawsooaa: 20214 rows
Total: 2884766


['dataset_name',
 'era',
 'genre',
 'location',
 'meter',
 'overall_explanation',
 'poem_id',
 'poem_language',
 'poem_text',
 'poem_title',
 'poem_type',
 'poem_url',
 'poem_verses',
 'poet_description',
 'poet_english_id',
 'poet_era',
 'poet_id',
 'poet_name',
 'poet_url',
 'rhyme',
 'source',
 'tags',
 'verses_explanation',
 'poem_text_no_diacritics',
 'poet_name_with_diacritics']

In [ ]:
import pandas as pd 

all_poems_df = pd.read_csv('../data/cleaned_train.tsv', sep='\t')

/tmp/ipykernel_2971615/50929735.py:3: DtypeWarning: Columns (1,2,3,4,7,9,10,11,13,14,15,18,19,20,21) have mixed types. Specify dtype option on import or set low_memory=False.
  all_poems_df = pd.read_csv('../data/cleaned_train.tsv', sep='\t')


In [ ]:
from IPython.display import display


print("Columns in DataFrame:")
columns_to_display = [col for col in all_poems_df.columns if col not in ['source', 'verse_count', 'explanation', 'poem_text', 'poem_text_no_diacritics']]
print(columns_to_display)
print("\n" + "="*80 + "\n")

for dataset_name, group_df in all_poems_df.groupby('dataset_name'):
    print(f"📘 Dataset: {dataset_name}")
    display(group_df[columns_to_display].head())  # Display only selected columns
    print("-" * 80)


Columns in DataFrame:
['dataset_name', 'era', 'genre', 'location', 'meter', 'overall_explanation', 'poem_id', 'poem_language', 'poem_title', 'poem_type', 'poem_url', 'poem_verses', 'poet_description', 'poet_english_id', 'poet_era', 'poet_id', 'poet_name', 'poet_url', 'rhyme', 'tags', 'verses_explanation', 'poet_name_with_diacritics']


📘 Dataset: AraPoems


,dataset_name,era,genre,location,meter,overall_explanation,poem_id,poem_language,poem_title,poem_type,...,poet_description,poet_english_id,poet_era,poet_id,poet_name,poet_url,rhyme,tags,verses_explanation,poet_name_with_diacritics
1904750,AraPoems,NaN,NaN,NaN,NaN,NaN,1911748,NaN,إيهاً جُدابُ سَيِّدَ الْأَعْرابِ,NaN,...,NaN,NaN,العصر الجاهلي,NaN,صفية بنت ثعلبة الشيبانية,NaN,NaN,NaN,NaN,صَفِيَّة بنت ثَعْلَبَة الشَّيْبانِيَّة
1904751,AraPoems,NaN,NaN,NaN,NaN,NaN,1911749,NaN,إِنَّ الْجُنُودَ حَثُّها طِلابُها,NaN,...,NaN,NaN,العصر الجاهلي,NaN,صفية بنت ثعلبة الشيبانية,NaN,NaN,NaN,NaN,صَفِيَّة بنت ثَعْلَبَة الشَّيْبانِيَّة
1904752,AraPoems,NaN,NaN,NaN,NaN,NaN,1911750,NaN,لَيْسَ لِلْعُجْمِ نُصْرَةٌ فِي عَشِيرِي,NaN,...,NaN,NaN,العصر الجاهلي,NaN,صفية بنت ثعلبة الشيبانية,NaN,NaN,NaN,NaN,صَفِيَّة بنت ثَعْلَبَة الشَّيْبانِيَّة
1904753,AraPoems,NaN,NaN,NaN,NaN,NaN,1911751,NaN,إن نصر الطميح أكرمُ نصرٍ,NaN,...,NaN,NaN,العصر الجاهلي,NaN,صفية بنت ثعلبة الشيبانية,NaN,NaN,NaN,NaN,صَفِيَّة بنت ثَعْلَبَة الشَّيْبانِيَّة
1904754,AraPoems,NaN,NaN,NaN,NaN,NaN,1911752,NaN,احْمِلْ ظَلِيمُ فِي الْعَجاجِ الْأَسْوَدِ,NaN,...,NaN,NaN,العصر الجاهلي,NaN,صفية بنت ثعلبة الشيبانية,NaN,NaN,NaN,NaN,صَفِيَّة بنت ثَعْلَبَة الشَّيْبانِيَّة


--------------------------------------------------------------------------------
📘 Dataset: Arabic PCD


,dataset_name,era,genre,location,meter,overall_explanation,poem_id,poem_language,poem_title,poem_type,...,poet_description,poet_english_id,poet_era,poet_id,poet_name,poet_url,rhyme,tags,verses_explanation,poet_name_with_diacritics
72980,Arabic PCD,NaN,NaN,NaN,بحر الطويل,NaN,79978,NaN,NaN,NaN,...,NaN,NaN,العصر الجاهلي,NaN,عمرو بن قميئة,NaN,د,NaN,NaN,عمرو بنِ قُمَيئَة
72981,Arabic PCD,NaN,NaN,NaN,بحر الطويل,NaN,79979,NaN,NaN,NaN,...,NaN,NaN,العصر الجاهلي,NaN,عمرو بن قميئة,NaN,د,NaN,NaN,عمرو بنِ قُمَيئَة
72982,Arabic PCD,NaN,NaN,NaN,بحر الطويل,NaN,79980,NaN,NaN,NaN,...,NaN,NaN,العصر الجاهلي,NaN,عمرو بن قميئة,NaN,د,NaN,NaN,عمرو بنِ قُمَيئَة
72983,Arabic PCD,NaN,NaN,NaN,بحر الطويل,NaN,79981,NaN,NaN,NaN,...,NaN,NaN,العصر الجاهلي,NaN,عمرو بن قميئة,NaN,د,NaN,NaN,عمرو بنِ قُمَيئَة
72984,Arabic PCD,NaN,NaN,NaN,بحر الطويل,NaN,79982,NaN,NaN,NaN,...,NaN,NaN,العصر الجاهلي,NaN,عمرو بن قميئة,NaN,د,NaN,NaN,عمرو بنِ قُمَيئَة


--------------------------------------------------------------------------------
📘 Dataset: Arabic Poetry Dataset


...

--------------------------------------------------------------------------------
📘 Dataset: mawsooaa


,dataset_name,era,genre,location,meter,overall_explanation,poem_id,poem_language,poem_title,poem_type,...,poet_description,poet_english_id,poet_era,poet_id,poet_name,poet_url,rhyme,tags,verses_explanation,poet_name_with_diacritics
2802866,mawsooaa,NaN,غير محدد,NaN,بحر الطويل,NaN,2809909,فصيح,خَلِيلَيَّ لَا تَسْتَعْجِلَا أَنْ تَزَوَّدا,NaN,...,NaN,NaN,NaN,NaN,عمرو بن قميئة,NaN,د,NaN,NaN,عَمرُو بنُ قَمِيئَة
2802867,mawsooaa,NaN,غير محدد,NaN,بحر الطويل,NaN,2809910,فصيح,أَرَى جَارَتِي خَفَّتْ وَخَفَّ نَصِيحُها,NaN,...,NaN,NaN,NaN,NaN,عمرو بن قميئة,NaN,ح,NaN,NaN,عَمرُو بنُ قَمِيئَة
2802868,mawsooaa,NaN,غير محدد,NaN,بحر الطويل,NaN,2809911,فصيح,إِنْ أَكُ قَدْ أَقْصَرْتُ عَنْ طُولِ رِحْلَةٍ,NaN,...,NaN,NaN,NaN,NaN,عمرو بن قميئة,NaN,م,NaN,NaN,عَمرُو بنُ قَمِيئَة
2802869,mawsooaa,NaN,غير محدد,NaN,بحر المنسرح,NaN,2809912,فصيح,يَا لَهْفَ نَفْسِي عَلَى الشَّبَابِ وَلَمْ,NaN,...,NaN,NaN,NaN,NaN,عمرو بن قميئة,NaN,م,NaN,NaN,عَمرُو بنُ قَمِيئَة
2802870,mawsooaa,NaN,غير محدد,NaN,بحر المتقارب,NaN,2809913,فصيح,تَحِنُّ حَنِيناً إِلَى مَالِكٍ,NaN,...,NaN,NaN,NaN,NaN,عمرو بن قميئة,NaN,ل,NaN,NaN,عَمرُو بنُ قَمِيئَة


--------------------------------------------------------------------------------
📘 Dataset: poems_hakim


,dataset_name,era,genre,location,meter,overall_explanation,poem_id,poem_language,poem_title,poem_type,...,poet_description,poet_english_id,poet_era,poet_id,poet_name,poet_url,rhyme,tags,verses_explanation,poet_name_with_diacritics
2392320,poems_hakim,NaN,قصيدة المعلقات,NaN,NaN,NaN,2399320,NaN,أمن أم أوفى دمنة لم تكلم,NaN,...,NaN,NaN,العصر الجاهلي,NaN,زهير بن أبي سلمى,NaN,NaN,NaN,NaN,زهير بن أبي سلمى
2392321,poems_hakim,NaN,قصيدة المعلقات,NaN,NaN,NaN,2399321,NaN,آذنتنا ببينها أسماء,NaN,...,NaN,NaN,العصر الجاهلي,NaN,الحارث بن حلزة,NaN,NaN,NaN,NaN,الحارث بن حلزة
2392322,poems_hakim,NaN,قصيدة المعلقات,NaN,NaN,NaN,2399322,NaN,قفا نبك من ذكرى حبيب ومنزِل,NaN,...,NaN,NaN,العصر الجاهلي,NaN,امرؤ القيس,NaN,NaN,NaN,NaN,امرؤ القيس
2392323,poems_hakim,NaN,قصيدة المعلقات,NaN,NaN,NaN,2399323,NaN,يا دار مية بالعلياء فالسند,NaN,...,NaN,NaN,العصر الجاهلي,NaN,النابغة الذبياني,NaN,NaN,NaN,NaN,النابغة الذبياني
2392324,poems_hakim,NaN,قصيدة المعلقات,NaN,NaN,NaN,2399324,NaN,أقفر من أهله ملحوب,NaN,...,NaN,NaN,العصر الجاهلي,NaN,عبيد بن الأبرص,NaN,NaN,NaN,NaN,عبيد بن الأبرص


--------------------------------------------------------------------------------


In [ ]:
import pandas as pd

# Group the final dataset by the 'source' column
grouped_sources = all_poems_df.groupby('dataset_name')

# Iterate through each group and calculate statistics
for source, group_df in grouped_sources:
    # Calculate the number of rows
    num_rows = len(group_df)
    
    # Calculate the average number of characters per row for 'poem_text'
    avg_chars_per_row = group_df['poem_text'].dropna().apply(len).mean()
    
    # Calculate the average number of verses
    avg_verses = group_df['poem_verses'].dropna().mean()
    
    # Get the columns with values (not None or NaN)
    columns_with_values = group_df.dropna(axis=1, how='all').columns.tolist()
    
    # Print the statistics
    print(f"Source: {source}")
    print(f"Number of rows: {num_rows}")
    print(f"Average characters per row (poem_text): {avg_chars_per_row:.2f}")
    print(f"Average number of verses: {avg_verses:.2f}")
    print(f"Columns with values: {columns_with_values}")
    print("-" * 80)

Source: AraPoems
Number of rows: 165509
Average characters per row (poem_text): 593.51
Average number of verses: 12.61
Columns with values: ['dataset_name', 'genre', 'poem_id', 'poem_text', 'poem_title', 'poem_verses', 'poet_era', 'poet_name', 'source', 'poem_text_no_diacritics', 'poet_name_with_diacritics']
--------------------------------------------------------------------------------
Source: Arabic PCD
Number of rows: 1831770
Average characters per row (poem_text): 56.47
Average number of verses: 1.00
Columns with values: ['dataset_name', 'meter', 'poem_id', 'poem_text', 'poem_verses', 'poet_era', 'poet_name', 'rhyme', 'poem_text_no_diacritics', 'poet_name_with_diacritics']
--------------------------------------------------------------------------------
Source: Arabic Poetry Dataset
Number of rows: 72980
Average characters per row (poem_text): 677.09
Average number of verses: 11.03
Columns with values: ['dataset_name', 'genre', 'meter', 'poem_id', 'poem_text', 'poem_title', 'poem_t